# Etapa 1: Análise exploratória dos dados e diagnóstico

**Disciplina:** Inteligência Artificial — Prof. Cristiane Neri Nobre
**Entrega:** 01/11/2026 | **Valor:** 2,00 pontos

Reaproveitamento do código anterior dos membros 1–4, adaptado a `treino.csv` e ao alvo binário.
As análises que estavam pendentes continuam como tarefas; não há treino de modelos nesta etapa.

| Responsável | Tarefa nesta adaptação |
|---|---|
| Membro 1 | Revisar carga, tipos e ausências da nova base; reescrever a síntese com os resultados atuais. |
| Membro 2 | Revisar distribuição normal/anomaly e justificar F1-macro, sem agrupamento em famílias. |
| Membro 3 | Revisar cardinalidade, espaços/caixa e ausências; atualizar a justificativa da codificação. |
| Membro 4 | Revisar duplicatas, conflitos, constantes e novos atributos derivados; propor comparações. |
| Membro 5 | Implementar associação de cada atributo com a classe, investigar vazamento e consolidar o diagnóstico. |

Substituir Membro 1–5 pelos nomes da equipe. Cada responsável deve executar e interpretar sua seção.
As conclusões da base antiga foram retiradas; os espaços de síntese devem ser preenchidos pela equipe.

In [ ]:
from pathlib import Path
from itertools import combinations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "data" / "treino.csv").exists():
    project_root = project_root.parent
if not (project_root / "data" / "treino.csv").exists():
    raise FileNotFoundError("Execute na raiz do projeto ou em notebooks/.")

# Leitura direta: os módulos src ainda se referem à base antiga.
CLASSES = ["normal", "anomaly"]
sns.set_theme(style="whitegrid")

## Membro 1 — Caracterização geral da base e análise dos tipos de dados

Usamos exclusivamente `treino.csv`, com cabeçalho e alvo `class`.
Preservamos todas as colunas para diagnóstico, inclusive os sinais adicionados que o Membro 5 investigará.
Não consultamos o teste e não modificamos os arquivos oficiais. Tratamentos serão comparados na etapa 2.

### 1.1 Carregamento do conjunto de treino bruto

In [ ]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
df_train = pd.read_csv(project_root / "data" / "treino.csv")
df_train.head()

### 1.2 Dimensões e papel de cada coluna

In [ ]:
COL_ROTULO = "class"
COLS_ATRIBUTOS = [c for c in df_train.columns if c != COL_ROTULO]
n_linhas, n_colunas = df_train.shape
assert n_linhas > 0
assert COL_ROTULO in df_train
assert df_train[COL_ROTULO].notna().all()
assert set(df_train[COL_ROTULO]) == set(CLASSES)
assert "id" not in COLS_ATRIBUTOS, "ID não deve ser tratado como preditor."
estrutura = pd.DataFrame({
    "Grupo": ["Atributos a investigar", "Alvo class", "Total de colunas"],
    "Quantidade": [len(COLS_ATRIBUTOS), 1, n_colunas],
})
print(f"Instâncias: {n_linhas:,}; colunas: {n_colunas}")
print(f"Uso de memória: {df_train.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(estrutura)

In [ ]:
print("Classes observadas:", sorted(df_train[COL_ROTULO].unique()))
print("Colunas:", df_train.columns.tolist())
# Não presumimos difficulty_level nem o esquema de 43 colunas do arquivo antigo.

### 1.3 Mapeamento dos tipos dos atributos

Distinguimos dois níveis:
- **Tipo físico:** como o pandas armazenou a coluna (inteiro, real ou texto).
- **Tipo semântico:** o que o valor representa (categórico, binário, contagem inteira, taxa real). Isso importa porque um atributo armazenado como inteiro pode ser, na prática, uma variável binária ou categórica, e o tratamento na Etapa 2 depende dessa leitura.

In [ ]:
def tipo_fisico(serie: pd.Series) -> str:
    if pd.api.types.is_integer_dtype(serie):
        return "inteiro"
    if pd.api.types.is_float_dtype(serie):
        return "real"
    return "categórico (texto)"

def tipo_semantico(serie: pd.Series) -> str:
    # Heurística exploratória; ausência pode fazer contagens inteiras serem armazenadas como float.
    if not pd.api.types.is_numeric_dtype(serie):
        return "categórico nominal"
    valores = set(serie.dropna().unique())
    if not valores:
        return "totalmente ausente"
    if len(valores) == 1:
        return "constante"
    if valores <= {0, 1}:
        return "binário (0/1)"
    if pd.api.types.is_float_dtype(serie) and serie.min() >= 0 and serie.max() <= 1:
        return "real – taxa em [0, 1]"
    if pd.api.types.is_integer_dtype(serie):
        return "inteiro – contagem/magnitude"
    return "real"

mapa_tipos = pd.DataFrame({
    "dtype_pandas": df_train[COLS_ATRIBUTOS].dtypes.astype(str),
    "tipo_fisico": [tipo_fisico(df_train[c]) for c in COLS_ATRIBUTOS],
    "tipo_semantico": [tipo_semantico(df_train[c]) for c in COLS_ATRIBUTOS],
    "n_valores_distintos": df_train[COLS_ATRIBUTOS].nunique(),
})
mapa_tipos

In [ ]:
# Resumo: quantos atributos em cada tipo físico e semântico
resumo_fisico = mapa_tipos["tipo_fisico"].value_counts().rename("qtd_atributos")
resumo_semantico = mapa_tipos["tipo_semantico"].value_counts().rename("qtd_atributos")
display(resumo_fisico.to_frame())
display(resumo_semantico.to_frame())

COLS_CATEGORICAS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "categórico (texto)"].tolist()
COLS_INTEIRAS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "inteiro"].tolist()
COLS_REAIS = mapa_tipos.index[mapa_tipos["tipo_fisico"] == "real"].tolist()
COLS_NUMERICAS = COLS_INTEIRAS + COLS_REAIS

print("Categóricos:", COLS_CATEGORICAS)
print(f"Inteiros ({len(COLS_INTEIRAS)}):", COLS_INTEIRAS)
print(f"Reais ({len(COLS_REAIS)}):", COLS_REAIS)

In [ ]:
# Hipótese de domínio a verificar; ausentes são contados separadamente.
# A interpretação de su_attempted deve ser confirmada antes de recodificar.
BINARIOS_DOCUMENTADOS = ["land", "logged_in", "root_shell", "su_attempted",
                         "is_host_login", "is_guest_login"]

verif_binarios = pd.DataFrame({
    "valores_observados": [sorted(df_train[c].dropna().unique().tolist()) for c in BINARIOS_DOCUMENTADOS],
    "ausentes": [df_train[c].isna().sum() for c in BINARIOS_DOCUMENTADOS],
    "frequencia_valor_1": [(df_train[c] == 1).sum() for c in BINARIOS_DOCUMENTADOS],
    "dominio_0_1_respeitado": [set(df_train[c].dropna().unique()) <= {0, 1} for c in BINARIOS_DOCUMENTADOS],
}, index=BINARIOS_DOCUMENTADOS)
verif_binarios

In [ ]:
# Cardinalidade dos atributos categóricos (relevante para a escolha da codificação na Etapa 2)
card = pd.DataFrame({
    "n_categorias": df_train[COLS_CATEGORICAS].nunique(),
    "categoria_mais_frequente": [df_train[c].mode()[0] for c in COLS_CATEGORICAS],
    "pct_mais_frequente": [df_train[c].value_counts(normalize=True).iloc[0] * 100 for c in COLS_CATEGORICAS],
    "categorias_com_<10_registros": [(df_train[c].value_counts() < 10).sum() for c in COLS_CATEGORICAS],
})
card

### 1.4 Valores ausentes

In [ ]:
# (a) Ausentes explícitos (NaN/None) por coluna, em todas as colunas do CSV
ausentes = pd.DataFrame({
    "n_ausentes": df_train.isna().sum(),
    "pct_ausentes": df_train.isna().mean() * 100,
})
print(f"Colunas com pelo menos um ausente: {(ausentes['n_ausentes'] > 0).sum()} de {n_colunas}")
print(f"Total de células ausentes: {int(ausentes['n_ausentes'].sum()):,} "
      f"de {df_train.size:,} células ({ausentes['n_ausentes'].sum() / df_train.size * 100:.4f}%)")
print(f"Linhas com pelo menos um ausente: {df_train.isna().any(axis=1).sum():,}")
ausentes.T

In [ ]:
# (b) Ausentes "disfarçados": marcadores que não viram NaN na leitura do arquivo.
# Em bases de rede é comum usar '?', '-', strings vazias ou valores negativos como "sem informação".
MARCADORES_TEXTO = {"?", "-", "", "na", "n/a", "nan", "null", "none"}
colunas_texto = COLS_CATEGORICAS + [COL_ROTULO]

marcadores = {
    c: df_train[c].dropna().astype(str).str.strip().str.lower().isin(MARCADORES_TEXTO).sum()
    for c in colunas_texto
}
negativos = (df_train[COLS_NUMERICAS] < 0).sum()

print("Marcadores textuais de ausência nas colunas categóricas/rótulo:", marcadores)
print("Colunas numéricas com valores negativos:", int((negativos > 0).sum()))

### 1.5 Estatísticas descritivas dos atributos numéricos

Além de média, desvio padrão e quartis, incluímos o coeficiente de variação, a assimetria e a proporção de zeros, que ajudam a justificar as propostas de transformação da Etapa 2.

In [ ]:
desc = df_train[COLS_NUMERICAS].describe().T  # count, mean, std, min, 25%, 50%, 75%, max
desc["cv"] = desc["std"] / desc["mean"].replace(0, np.nan)        # dispersão relativa
desc["assimetria"] = df_train[COLS_NUMERICAS].skew()
desc["pct_zeros"] = (df_train[COLS_NUMERICAS] == 0).mean() * 100
desc.insert(0, "tipo", mapa_tipos.loc[COLS_NUMERICAS, "tipo_semantico"])
desc

In [ ]:
# Destaques objetivos extraídos da tabela acima
print("Atributos com desvio padrão zero (constantes):",
      desc.index[desc["std"] == 0].tolist())
print("\nAtributos com mediana = 0 e Q3 = 0 (entre os valores observados; conferir pct_zeros):")
print(desc.index[(desc["50%"] == 0) & (desc["75%"] == 0)].tolist())
print("\nMaiores amplitudes (max - min):")
print((desc["max"] - desc["min"]).sort_values(ascending=False).head(5).apply(lambda v: f"{v:,.0f}"))
print("\nAtributos com |assimetria| > 10:")
print(desc.loc[desc["assimetria"].abs() > 10, "assimetria"].sort_values(ascending=False).round(1))

### 1.6 Observações estruturais para os demais membros

Verificação rápida de redundância e conflito de rótulos. Não tratamos nada aqui: os números servem de ponto de partida para quem analisar dados redundantes/inconsistentes.

In [ ]:
dup_total = df_train.duplicated().sum()
# Comparação diagnóstica com todos os atributos disponíveis; não remove linhas.
grupos = df_train.groupby(COLS_ATRIBUTOS, dropna=False)[COL_ROTULO].nunique()
grupos_conflitantes = grupos[grupos > 1]
linhas_conflitantes = df_train.merge(
    grupos_conflitantes.reset_index()[COLS_ATRIBUTOS], on=COLS_ATRIBUTOS, how="inner"
)
print(f"Duplicatas exatas: {dup_total}")
print(f"Grupos com atributos idênticos e classes distintas: {len(grupos_conflitantes)}")
display(linhas_conflitantes[["protocol_type", "service", "flag", "src_bytes", "dst_bytes", COL_ROTULO]].head(12))

### 1.7 Síntese do Membro 1 — atualizar após executar

Descrever dimensões, tipos físicos versus semânticos, ausências e diferenças de escala com os números atuais.
Não reutilizar a conclusão antiga de que não há ausentes. Separar constância entre valores observados de ausência.
Propor tratamentos para comparação na etapa 2, sem ajustá-los agora.

**Texto da equipe:** a preencher.

## Membro 2 — Classes e severidade do desbalanceamento

O alvo oficial já é binário: `normal` e `anomaly`. Não há informação para decompor anomaly em famílias.
Reutilizamos as funções de contagem e visualização, sem remapear ou reamostrar o alvo.

### 2.1 Classes oficiais: identificação e frequências

In [ ]:
df_classes = df_train[[COL_ROTULO]].copy()
rotulos_originais = df_train[COL_ROTULO].copy(deep=True)
n_treino = len(df_classes)
assert set(df_classes[COL_ROTULO]) == set(CLASSES)
print("Classes oficiais:", CLASSES)

def tabela_frequencias(serie):
    """Conta cada categoria e calcula seu percentual sobre todas as linhas recebidas."""
    tabela = serie.value_counts(dropna=False).rename("frequencia").to_frame()
    tabela["percentual"] = 100 * tabela["frequencia"] / len(serie)
    assert tabela["frequencia"].sum() == len(serie), "Perda de registros na contagem."
    assert np.isclose(tabela["percentual"].sum(), 100), "Percentuais não somam 100%."
    return tabela


freq_rotulos = tabela_frequencias(df_classes[COL_ROTULO])
display(freq_rotulos)

### 2.2 Distribuição binária

Mantemos os gráficos em escalas linear e logarítmica para comparar a frequência das duas classes.

In [ ]:
# As mesmas cores identificam cada classe nas duas escalas.
cores_classes = dict(zip(CLASSES, sns.color_palette("colorblind", len(CLASSES))))


def plotar_distribuicao(tabela, classes, titulo, altura):
    """Mostra frequências lineares e logarítmicas sem ocultar categorias raras."""
    fig, eixos = plt.subplots(1, 2, figsize=(15, altura), sharey=True, layout="constrained")
    posicoes = np.arange(len(tabela))
    valores = tabela["frequencia"].to_numpy()
    cores = [cores_classes[classe] for classe in classes]
    for eixo in eixos:
        eixo.barh(posicoes, valores, color=cores)
        eixo.set_yticks(posicoes, tabela.index)
        eixo.grid(axis="y", visible=False)
        eixo.set_axisbelow(True)
    eixos[0].invert_yaxis()
    eixos[0].set_xlim(0, valores.max() * 1.55)
    eixos[0].set_title("Escala linear: concentração dos registros")
    eixos[0].set_xlabel("Número de registros")
    for posicao, (quantidade, percentual) in enumerate(
        tabela[["frequencia", "percentual"]].itertuples(index=False, name=None)
    ):
        eixos[0].annotate(
            f"{quantidade:,.0f} ({percentual:.4f}%)", (quantidade, posicao),
            xytext=(5, 0), textcoords="offset points", va="center", fontsize=9,
        )
    eixos[1].set_xscale("log")
    eixos[1].set_xlim(1, valores.max() * 2)
    eixos[1].set_title("Escala logarítmica: visibilidade das categorias raras")
    eixos[1].set_xlabel("Número de registros (log base 10)")
    fig.suptitle(f"{titulo} | treino.csv (n = {n_treino:,})", fontsize=14)
    plt.show()


plotar_distribuicao(freq_rotulos, freq_rotulos.index, "Classes oficiais", altura=4.5)

### 2.3 Diagnóstico quantitativo do desbalanceamento

Comparar maioria e minoria. Discutir por que o F1-macro é preferível à acurácia isolada.

In [ ]:
razao_rotulos = freq_rotulos["frequencia"].max() / freq_rotulos["frequencia"].min()
print(f"Razão maioria/minoria: {razao_rotulos:,.2f}:1")
print(f"Acurácia de sempre prever a maioria: {freq_rotulos['percentual'].max():.2f}%")
print(f"Proporção anomaly: {freq_rotulos.loc['anomaly', 'percentual']:.2f}%")
assert df_train[COL_ROTULO].equals(rotulos_originais), "O alvo original foi alterado."

### 2.4 Síntese do Membro 2

**A preencher:** apresentar quantidades e proporções atuais, identificar a minoria, explicar o peso igual
por classe no F1-macro e propor duas técnicas de balanceamento para testar na etapa 2.
Não usar estatísticas de ataques específicos ou famílias da base antiga.

## Membro 3 — Atributos categóricos e alta cardinalidade

Revisar o código reaproveitado e acrescentar a investigação de diferenças de caixa/espaços e ausências.
Justificar a codificação de service e o tratamento de categorias desconhecidas sem ajustar encoders nesta etapa.

### 3.1 Identificação e cardinalidade

Analisar todos os atributos textuais do treino, incluindo os adicionados. Sua presença aqui não autoriza usá-los nos modelos.

In [ ]:
# Reaproveita a análise de cardinalidade para todos os categóricos do CSV.
categoricos_m3 = COLS_CATEGORICAS.copy()
dados_categoricos_m3 = df_train[categoricos_m3].copy()

cardinalidade_m3 = pd.DataFrame({"tipo": dados_categoricos_m3.dtypes.astype(str),"categorias_distintas": dados_categoricos_m3.nunique(),"valores_ausentes": dados_categoricos_m3.isna().sum(),})
cardinalidade_m3.index.name = "atributo"

display(cardinalidade_m3)

for atributo in categoricos_m3:
    categorias = sorted(dados_categoricos_m3[atributo].dropna().unique())
    print(f"\n{atributo} ({len(categorias)} categorias):")
    print(", ".join(categorias))

### 3.2 Distribuição dos atributos categóricos

Calculamos as frequências absolutas e percentuais das categorias no conjunto
de treino. Os percentuais usam o total de registros do treino como denominador.
As tabelas permitem identificar categorias dominantes e pouco representadas.

In [ ]:
# Guarda as distribuições para reutilização nas próximas análises.
frequencias_m3 = {}

for atributo in categoricos_m3:
    tabela = (dados_categoricos_m3[atributo].value_counts(dropna=False).rename_axis("categoria").to_frame("frequencia"))
    tabela["percentual"] = (tabela["frequencia"] / len(dados_categoricos_m3) * 100)
    frequencias_m3[atributo] = tabela

    # Confere se todos os registros foram contabilizados.
    assert tabela["frequencia"].sum() == len(dados_categoricos_m3)
    assert np.isclose(tabela["percentual"].sum(), 100)
    print(f"\nDistribuição de {atributo}:")
    with pd.option_context("display.max_rows", None):
        display(tabela)

### 3.3 Serviços frequentes, raros e cauda longa

Analisamos a concentração dos registros nos dez serviços mais frequentes
e a presença de serviços com poucas observações.

Para este diagnóstico, consideramos raros os serviços com menos de 100
registros no treino, aproximadamente 0,08% da base por serviço. Esse limiar
é uma escolha exploratória para descrever categorias pouco representadas,
não uma regra de agrupamento já definida para o pré-processamento. A cauda longa é avaliada comparando a quantidade de categorias raras
com a proporção de registros que elas representam.

In [ ]:
#Reutiliza as frequências calculadas na seção 3.2.
servicos_m3 = frequencias_m3["service"].loc[frequencias_m3["service"].index.notna()].copy()
# Ausência não é um serviço; percentuais continuam sobre todas as linhas do treino.
limiar_raridade_m3 = 100
top10_m3 = servicos_m3.head(10)
raros_m3 = servicos_m3[servicos_m3["frequencia"] < limiar_raridade_m3]

print("Dez serviços mais frequentes:")
display(top10_m3)
print(f"\nServiços raros: menos de {limiar_raridade_m3} registros:")
display(raros_m3)

#quantifica a concentração e o tamanho da cauda longa
n_servicos_m3 = len(servicos_m3)
n_raros_m3 = len(raros_m3)
registros_raros_m3 = int(raros_m3["frequencia"].sum())

print(f"\nOs 10 serviços mais frequentes representam "f"{top10_m3['percentual'].sum():.2f}% dos registros.")
print(f"Serviços raros: {n_raros_m3} de {n_servicos_m3} categorias "f"({100 * n_raros_m3 / n_servicos_m3:.2f}% das categorias).")
print(f"Esses serviços somam {registros_raros_m3:,} registros "f"({raros_m3['percentual'].sum():.4f}% do treino).")

#Visualiza os serviços dominantes e a distribuição completa.
fig, eixos = plt.subplots(1, 2, figsize=(15, 5), layout="constrained")
top10_m3["frequencia"].sort_values().plot.barh(ax=eixos[0], color="steelblue")
eixos[0].set_title("Dez serviços mais frequentes")
eixos[0].set_xlabel("Número de registros")
eixos[0].set_ylabel("Serviço")
eixos[0].grid(axis="y", visible=False)
posicoes_m3 = np.arange(1, n_servicos_m3 + 1)
eixos[1].plot(posicoes_m3,servicos_m3["frequencia"].to_numpy(),marker=".",color="steelblue",)
eixos[1].axhline(limiar_raridade_m3,color="darkred",linestyle="--",label="Limiar exploratório: 100 registros",)
eixos[1].set_yscale("log")
eixos[1].set_title("Distribuição completa dos serviços")
eixos[1].set_xlabel("Posição do serviço, do mais ao menos frequente")
eixos[1].set_ylabel("Número de registros (escala logarítmica)")
eixos[1].legend()

plt.show()

### Revisão pendente — Membro 3

Implementar a comparação de cardinalidade antes/depois de remover espaços e uniformizar caixa em uma cópia.
Distinguir categoria rara de ausência. Confirmar com as saídas acima a concentração dos dez serviços e a cauda longa.
Não fixar os números de categorias observados no dataset antigo.

### 3.4 Discussão sobre alternativas de codificação e categorias desconhecidas

Os atributos analisados possuem categorias sem uma ordem natural. Por isso, substituir seus valores por números como 1, 2 e 3 pode sugerir uma ordem que não existe. O One-Hot Encoding evita isso criando uma coluna para cada categoria. A quantidade de colunas deve ser recalculada a partir da cardinalidade atual e da estratégia para ausentes. Serviços raros geram colunas com poucos valores 1.

O Count Encoding substitui cada categoria pela sua quantidade de ocorrências, enquanto o Frequency Encoding utiliza sua frequência relativa. Essas alternativas mantêm uma coluna por atributo, mas categorias com frequências iguais recebem o mesmo valor, perdendo sua distinção. Outra possibilidade é agrupar serviços raros em uma categoria comum, reduzindo a quantidade de colunas, com o risco de juntar serviços com comportamentos diferentes. O limite de 100 registros utilizado nesta análise serve apenas para identificar raridade, sem definir um agrupamento obrigatório.

Para categorias desconhecidas em dados futuros, o One-Hot Encoding pode ser configurado para deixar todas as colunas daquele atributo com valor zero. No Count/Frequency Encoding, é possível adotar zero como valor padrão para categorias não encontradas. Também podemos definir uma categoria específica para valores desconhecidos. Em todos os casos, as categorias e frequências devem ser obtidas apenas dos dados usados para treinamento, sem utilizar informações da validação. Nesta análise, não foi necessário carregar ou consultar o conjunto de teste oficial.

### 3.5 Síntese do Membro 3

**A preencher:** cardinalidades atuais, inconsistências de escrita, ausências e justificativa da codificação. Distinguir diagnóstico da implementação que será comparada na etapa 2.

## Membro 4 — Redundâncias, constantes e inconsistências

### 4.1 Duplicatas exatas e duplicatas de atributos

Reutilizamos os critérios de linhas completas e atributos sem classe. Aqui entram todos os atributos disponíveis.
**Revisão pendente:** repetir o diagnóstico em uma cópia após normalizar categorias e excluir eventuais proxies
identificados pelo Membro 5. Colunas operacionais e derivados podem mascarar duplicatas do tráfego original.
Manter cópias de um mesmo registro na mesma dobra evita contaminação da validação; não alterar o teste oficial.

In [ ]:
COL_ROTULO_M4 = COL_ROTULO
ATRIB_M4 = COLS_ATRIBUTOS.copy()
n_treino_m4 = len(df_train)
n_dup_total = int(df_train.duplicated().sum())
n_dup_atrib_rotulo = int(df_train.duplicated(subset=ATRIB_M4 + [COL_ROTULO_M4]).sum())
mascara_dup_parcial = df_train.duplicated(subset=ATRIB_M4, keep=False)
n_linhas_dup_parcial = int(mascara_dup_parcial.sum())
n_excedentes_dup_parcial = int(df_train.duplicated(subset=ATRIB_M4).sum())
resumo_dup_m4 = pd.DataFrame({
    "criterio": ["Exata: todas as colunas", "Atributos iguais: todas as linhas dos grupos", "Atributos iguais: excedentes"],
    "n_linhas": [n_dup_total, n_linhas_dup_parcial, n_excedentes_dup_parcial],
})
resumo_dup_m4["pct_do_treino"] = 100 * resumo_dup_m4.n_linhas / n_treino_m4
display(resumo_dup_m4)

### 4.2 Atributos iguais com classes divergentes

Reaproveitamos o agrupamento e o cálculo dos registros fora da classe majoritária de cada grupo,
agora somente para normal/anomaly. É um limite de erro de resubstituição para esses atributos,
não uma estimativa do erro em dados futuros.

In [ ]:
sub_dup_m4 = df_train.loc[mascara_dup_parcial, ATRIB_M4 + [COL_ROTULO_M4]].copy()
def _maior_contagem(s):
    return int(s.value_counts().iloc[0])

grupos_m4 = (sub_dup_m4.groupby(ATRIB_M4, sort=False, dropna=False)
    .agg(n_registros=(COL_ROTULO_M4, "size"), n_rotulos=(COL_ROTULO_M4, "nunique"),
         rotulos=(COL_ROTULO_M4, lambda s: " | ".join(sorted(s.unique()))),
         maior_rotulo=(COL_ROTULO_M4, _maior_contagem)).reset_index())
conflitos_m4 = grupos_m4[grupos_m4.n_rotulos > 1].copy()
n_grupos_conflito = len(conflitos_m4)
n_linhas_conflito = int(conflitos_m4.n_registros.sum())
erro_min_rotulos = int((conflitos_m4.n_registros - conflitos_m4.maior_rotulo).sum())
print(f"Grupos conflitantes: {n_grupos_conflito}; linhas: {n_linhas_conflito}")
print(f"Erro mínimo de resubstituição para esses atributos: {erro_min_rotulos} registros")
display(conflitos_m4[["protocol_type", "service", "flag", "n_registros", "rotulos"]])
linhas_conflito_m4 = sub_dup_m4.merge(conflitos_m4[ATRIB_M4], on=ATRIB_M4, how="inner")
afetados_m4 = linhas_conflito_m4[COL_ROTULO_M4].value_counts().rename("linhas_em_conflito").to_frame()
afetados_m4["total_no_treino"] = df_train[COL_ROTULO_M4].value_counts().reindex(afetados_m4.index)
afetados_m4["pct_da_classe"] = 100 * afetados_m4.linhas_em_conflito / afetados_m4.total_no_treino
assert len(linhas_conflito_m4) == n_linhas_conflito
display(afetados_m4)

### 4.3 Atributos constantes (variância zero)

Um atributo constante tem variância zero, ou seja, apresenta o mesmo valor em todas as linhas e não ajuda a distinguir classes. Além disso, pode causar problemas em normalizações.

Verificamos isso contando os valores únicos nos atributos do CSV e calculando o desvio padrão dos numéricos. Também buscamos colunas idênticas para identificar atributos redundantes.

In [ ]:
# 4.3 Atributos constantes (variância zero) e colunas idênticas entre si.

numericas_m4 = df_train[ATRIB_M4].select_dtypes(include="number").columns.tolist()

perfil_var_m4 = pd.DataFrame({
    "ausentes": df_train[ATRIB_M4].isna().sum(),
    "n_valores_unicos": df_train[ATRIB_M4].nunique(),
    "desvio_padrao": df_train[numericas_m4].std(),      # NaN nos categóricos
    "variancia": df_train[numericas_m4].var(),
})

constantes_unicos = set(perfil_var_m4.index[perfil_var_m4["n_valores_unicos"] == 1])
constantes_std = set(perfil_var_m4.index[perfil_var_m4["desvio_padrao"] == 0])
assert constantes_std <= constantes_unicos, "Divergência entre os dois critérios"
constantes_m4 = sorted(constantes_unicos)

print(f"Atributos com 1 único valor (entre valores observados): {constantes_m4}")
print(f"Atributos numéricos com desvio padrão = 0: {sorted(constantes_std)}")
if constantes_m4:
    display(pd.DataFrame({
        "valor_unico": [df_train[c].dropna().iloc[0] for c in constantes_m4],
        "n_registros": n_treino_m4,
    }, index=constantes_m4))

print("\nAtributos com menos valores distintos:")
display(perfil_var_m4.sort_values("n_valores_unicos").head(8))

# Colunas idênticas entre si (redundância exata de atributos)
pares_identicos_m4 = [(a, b) for a, b in combinations(ATRIB_M4, 2)
                      if df_train[a].equals(df_train[b])]
print(f"\nPares de colunas idênticas entre os atributos do CSV: {len(pares_identicos_m4)}")
for a, b in pares_identicos_m4:
    print(f"  {a} == {b}")

### 4.4 Quase constantes, concentração em zero e domínio de su_attempted

Mantemos os limiares exploratórios de 99%, 90% e 75%. A concentração do valor dominante é calculada
**entre observações não ausentes**, com o percentual ausente separado. Ausentes não contam como valores raros.
Comparar a proporção de anomaly fora do valor dominante com a proporção atual do treino.
Verificar su_attempted sem assumir que ausência ou valor 2 deva ser convertido automaticamente para 1.

In [ ]:
# 4.4 Quase constantes, concentração em zero e domínio de su_attempted.
LIMIAR_DOMINANTE_M4 = 99.0     # quase constante: >= 99% dos registros no mesmo valor
LIMIAR_CONCENTRADO_M4 = 90.0   # concentrado: entre 90% e 99% no mesmo valor
LIMIAR_ZEROS_M4 = 75.0         # apenas para listar atributos muito concentrados em zero

def perfil_concentracao(serie):
    contagem = serie.value_counts()
    if contagem.empty:
        return {"valor_dominante": np.nan, "pct_dominante": np.nan, "n_fora_do_dominante": 0}
    freq1 = int(contagem.iloc[0])
    return {
        "valor_dominante": contagem.index[0],
        "pct_dominante": 100 * freq1 / serie.notna().sum(),
        "n_fora_do_dominante": int(serie.notna().sum()) - freq1,
    }

conc_m4 = pd.DataFrame([{"atributo": c, **perfil_concentracao(df_train[c])} for c in ATRIB_M4]).set_index("atributo")
conc_m4["pct_zeros"] = (df_train[numericas_m4] == 0).mean() * 100      # NaN nos categóricos

conc_m4["pct_ausentes"] = df_train[ATRIB_M4].isna().mean() * 100
eh_constante = (conc_m4["n_fora_do_dominante"] == 0) & (conc_m4["pct_ausentes"] == 0)
eh_q99 = conc_m4["pct_dominante"] >= LIMIAR_DOMINANTE_M4
eh_conc = conc_m4["pct_dominante"] >= LIMIAR_CONCENTRADO_M4
conc_m4["categoria"] = np.select(
    [eh_constante, eh_q99, eh_conc],
    ["constante", "quase constante (>= 99% no mesmo valor)", "concentrado (90% a 99% no mesmo valor)"],
    default="sem concentração relevante",
)

candidatos_nzv_m4 = conc_m4.index[~conc_m4["categoria"].isin(["constante", "sem concentração relevante"])].tolist()

print(conc_m4["categoria"].value_counts().to_string())
print(f"\nAtributos com >= {LIMIAR_ZEROS_M4:.0f}% de zeros: {int((conc_m4['pct_zeros'] >= LIMIAR_ZEROS_M4).sum())}")

mostrar_m4 = conc_m4[(conc_m4["categoria"] != "sem concentração relevante")
                     | (conc_m4["pct_zeros"] >= LIMIAR_ZEROS_M4)]
display(mostrar_m4.sort_values("pct_dominante", ascending=False))

eh_ataque_m4 = df_train[COL_ROTULO_M4] != "normal"
base_ataque_m4 = 100 * eh_ataque_m4.mean()
linhas_sinal = []
for c in candidatos_nzv_m4:
    fora = df_train[c].notna() & (df_train[c] != conc_m4.loc[c, "valor_dominante"])
    if not fora.any():
        continue
    rotulos_fora = df_train.loc[fora, COL_ROTULO_M4].value_counts(normalize=True)
    linhas_sinal.append({
        "atributo": c,
        "n_fora_do_dominante": int(fora.sum()),
        "pct_ataque_fora_do_dominante": 100 * eh_ataque_m4[fora].mean(),
        "rotulo_mais_comum_fora": rotulos_fora.index[0],
        "pct_desse_rotulo_fora": 100 * rotulos_fora.iloc[0],
    })
sinal_nzv_m4 = pd.DataFrame(linhas_sinal, columns=["atributo", "n_fora_do_dominante", "pct_ataque_fora_do_dominante", "rotulo_mais_comum_fora", "pct_desse_rotulo_fora"]).set_index("atributo")
print(f"\nBase inteira: {base_ataque_m4:.2f}% dos registros são ataques.")
display(sinal_nzv_m4.sort_values("n_fora_do_dominante"))

# Domínio de su_attempted
dist_su_m4 = df_train["su_attempted"].value_counts(dropna=False).sort_index().rename("frequencia").to_frame()
dist_su_m4["percentual"] = 100 * dist_su_m4["frequencia"] / n_treino_m4
n_su_fora_dominio = int((df_train["su_attempted"].notna() & ~df_train["su_attempted"].isin([0, 1])).sum())
print("\nDistribuição de su_attempted:")
display(dist_su_m4)
print(f"Registros com su_attempted fora de {{0, 1}}: {n_su_fora_dominio}")
if n_su_fora_dominio:
    display(df_train.loc[df_train["su_attempted"].notna() & ~df_train["su_attempted"].isin([0, 1]), COL_ROTULO_M4]
            .value_counts().rename("registros_com_su_attempted_fora_do_dominio").to_frame())

### 4.5 Propostas para a etapa 2 — revisar com a equipe

A tabela abaixo reaproveita os diagnósticos para listar propostas, sem executar remoções ou correções.
Comparar tratamentos de redundância/inconsistência na etapa 2, dentro da validação no treino.
Não descartar atributos apenas por correlação exploratória ou poucos valores raros.

**Complemento do Membro 4:** verificar relações entre src_bytes/src_bytes_kb, dst_bytes/dst_bytes_kb,
total_bytes, same_srv_rate/same_srv_pct e protocol_type/protocolo_cod, considerando valores e padrões de ausência.

In [ ]:
decisoes = []
for c in constantes_m4:
    decisoes.append({"alvo": c, "proposta": "Avaliar remoção de constante",
                     "evidência": f"Um valor observado; {df_train[c].isna().sum()} ausentes. Confirmar em cada dobra."})
if n_su_fora_dominio:
    decisoes.append({"alvo": "su_attempted", "proposta": "Confirmar domínio antes de recodificar",
                     "evidência": f"{n_su_fora_dominio} valores observados fora de 0/1; ausentes separados."})
for c in candidatos_nzv_m4:
    decisoes.append({"alvo": c, "proposta": "Investigar antes de remover",
                     "evidência": f"{conc_m4.loc[c, 'pct_dominante']:.2f}% de concentração entre não ausentes."})
decisoes.extend([
    {"alvo": "duplicatas", "proposta": "Definir tratamento e manter cópias no mesmo fold",
     "evidência": f"{n_dup_total} duplicatas exatas; {n_linhas_dup_parcial} linhas com atributos repetidos."},
    {"alvo": "classes conflitantes", "proposta": "Documentar sem inventar rótulos",
     "evidência": f"{n_grupos_conflito} grupos / {n_linhas_conflito} linhas no diagnóstico atual."},
])
decisoes_m4 = pd.DataFrame(decisoes)
display(decisoes_m4)

### 4.6 Síntese do Membro 4

**A preencher após executar:** resultados atuais, novas redundâncias investigadas, problemas de domínio e estratégias a comparar na etapa 2. Não reutilizar os números de conflitos e conclusões de remoção da base antiga.

## Membro 5 — Investigação de vazamento e diagnóstico final

Esta parte ainda não estava implementada no notebook antigo e permanece a cargo do responsável.

1. Investigar a associação de **cada atributo** com class antes de qualquer modelo: numéricos, categóricos e padrões de ausência. Registrar constantes e limitações das medidas.
2. Analisar `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min`: origem, momento em que estariam disponíveis e possível dependência da classificação/triagem. Correlação não comprova vazamento; justificar qualquer exclusão.
3. Revisar correlação entre atributos e relações determinísticas com o Membro 4. Não procurar difficulty_level se não existe no CSV.
4. Consolidar as cinco análises e o plano de validação no treino. Não acessar o teste sem rótulos para escolher tratamentos ou tentar reconstruir seu gabarito.

In [ ]:
# TODO Membro 5: investigar os sinais operacionais adicionados e seus padrões de ausência.
# Registrar evidências e hipóteses de disponibilidade, sem exclusão automática por correlação.

In [ ]:
# TODO Membro 5: calcular associação de TODOS os atributos com class.
# Usar medidas adequadas a numéricos/categóricos e revisar redundâncias com o Membro 4.

### Sintese do Membro 5 e Diagnostico Consolidado da Etapa 1
*[Espaco para o Membro 5 redigir a conclusao sobre prevencao de data leakage e consolidar o diagnostico geral da equipe para entrega]*